In [63]:
print("hello")

hello


In [64]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic


load_dotenv()
# model = "claude-haiku-4-5"
base_url = "http://127.0.0.1:8045"
api_key = "sk-5b2137d3a6c74956a519669d86aa4e71"
client = Anthropic(api_key=api_key, base_url=base_url)
model = "gemini-3-flash"
# model = "claude-opus-4-5-thinking"

# Helper functions
from anthropic.types import Message


def add_user_message(messages, message):
    user_message = {
        "role": "user",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(user_message)


def add_assistant_message(messages, message):
    assistant_message = {
        "role": "assistant",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(assistant_message)



def chat(
    messages, 
    system=None, 
    temperature=1.0, 
    stop_sequences=[], 
    max_tokens=4000, 
    tools=None,
    tool_choice=None,
):
    params = {
        "model": model,
        "max_tokens": max_tokens,  # 增加到 4000，支持更长的输出
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }
    if tools:
        params["tools"] = tools
        
    if tool_choice:
        params["tool_choice"] = tool_choice


    if system:
        params["system"] = system

    message = client.messages.create(**params)
    # text = message.content[0].text
    # print(f"""chat text: {text_from_message(message=message)}""")
    # 客户端处理停止序列（代理可能不支持）
    # if stop_sequences:
    #     for stop_seq in stop_sequences:
    #         if stop_seq in text:
    #             text = text.split(stop_seq)[0]
    #             break

    return message

def text_from_message(message):
    return "\n".join([block.text for block in message.content if block.type == "text"])

In [ ]:
# Tools function

from datetime import datetime, timedelta
from dateutil.relativedelta import relativedelta

# get_current_datetime tool function
from anthropic.types import ToolParam

def get_current_datetime(date_format="%Y-%m-%d %H:%M:%S"):
    if not date_format:
        raise ValueError("date_format cannot be empty")
    return datetime.now().strftime(date_format)


def add_duration_to_datetime(
    datetime_str, duration=0, unit="days", input_format="%Y-%m-%d"
):
    """
    在给定日期时间上加上指定时长，并返回格式化后的新日期时间字符串。
    使用 dateutil.relativedelta 自动处理所有边界情况（月末、闰年等）。
    
    参数:
    - datetime_str: 输入的日期时间字符串
    - duration: 要添加的时间量，可以是正数（用于未来日期）或负数（用于过去日期）
    - unit: 时间单位，可以是 'seconds'、'minutes'、'hours'、'days'、'weeks'、'months' 或 'years'
    - input_format: 输入日期时间的格式字符串，使用 Python 的 strptime 格式代码
    
    返回:
    - 格式化后的新日期时间字符串
    """
    date = datetime.strptime(datetime_str, input_format)

    # 使用 relativedelta 统一处理所有时间单位，自动处理边界情况
    unit_map = {
        "seconds": "seconds",
        "minutes": "minutes",
        "hours": "hours",
        "days": "days",
        "weeks": "weeks",
        "months": "months",
        "years": "years",
    }
    
    if unit not in unit_map:
        raise ValueError(f"Unsupported time unit: {unit}. Must be one of: {list(unit_map.keys())}")
    
    # relativedelta 会自动处理所有复杂的边界情况：
    # - 月份加减时的年份进位
    # - 月末日期（如 1月31日 + 1个月 → 2月28/29日）
    # - 闰年处理
    # - 年份加减时的日期保持
    delta_kwargs = {unit_map[unit]: duration}
    new_date = date + relativedelta(**delta_kwargs)

    return new_date.strftime("%A, %B %d, %Y %I:%M:%S %p")


def set_reminder(content, timestamp):
    print(f"----\n 设置提醒 {timestamp}:\n{content}\n----")



pass

In [66]:
nowTime = get_current_datetime()
print("nowTime:",nowTime)
nextTime = add_duration_to_datetime(nowTime, 10, "days", "%Y-%m-%d %H:%M:%S")
set_reminder("吃药", nextTime)

nowTime: 2026-02-13 14:41:01
----
 设置提醒 Monday, February 23, 2026 02:41:01 PM:
吃药
----


In [67]:
# Tools schemas
get_current_datetime_schema = ToolParam(
    {
        "name": "get_current_datetime",
        "description": """根据指定格式返回当前日期和时间。
                此工具提供格式化的系统时间字符串，适用于时间戳记录、时间差计算或向用户显示当前时间。
                默认格式为 YYYY-MM-DD HH:MM:SS。""",
        "input_schema": {
            "type": "object",
            "properties": {
                "date_format": {
                    "type": "string",
                    "description": """指定返回日期时间格式的字符串，使用 Python 的 strftime 格式代码。
                    例如：'%Y-%m-%d' 返回日期（YYYY-MM-DD），'%H:%M:%S' 返回时间（HH:MM:SS），'%B %d, %Y' 返回如 'May 07, 2025' 的日期。
                    默认为 '%Y-%m-%d %H:%M:%S'，返回完整时间戳如 '2025-05-07 14:32:15'。.""",
                    "default": "%Y-%m-%d %H:%M:%S",
                }
            },
            "required": [],
        },
    }
)


add_duration_to_datetime_schema = {
    "name": "add_duration_to_datetime",
    "description": """
              将指定的时间长度添加到日期时间字符串，并以详细格式返回结果日期时间。
              此工具将输入的日期时间字符串转换为 Python datetime 对象，
              在请求的时间单位中添加指定的时间长度，并返回格式化后的结果日期时间字符串。
              它处理各种时间单位，包括秒、分钟、小时、天、周、月和年，对月份和年份的计算进行特殊处理，以考虑不同月份长度和闰年。
              输出始终以详细格式返回，包括星期几、月份名称、日期、年份和带 AM/PM 指示符的时间（
              例如，'Thursday, April 03, 2025 10:30:00 AM'）。
            """,
    "input_schema": {
        "type": "object",
        "properties": {
            "datetime_str": {
                "type": "string",
                "description": "要添加时间长度的输入日期时间字符串。应根据 input_format 参数进行格式化。",
            },
            "duration": {
                "type": "number",
                "description": "要添加到日期时间的时间量。可以是正数（用于未来日期）或负数（用于过去日期）。默认为 0。",
            },
            "unit": {
                "type": "string",
                "description": "时间长度的单位。必须是以下之一：'seconds'、'minutes'、'hours'、'days'、'weeks'、'months' 或 'years'。默认为 'days'。",
            },
            "input_format": {
                "type": "string",
                "description": "用于解析输入 datetime_str 的格式字符串，使用 Python 的 strptime 格式代码。例如，'%Y-%m-%d' 用于 ISO 格式日期，如 '2025-04-03'。默认为 '%Y-%m-%d'。",
            },
        },
        "required": ["datetime_str"],
    },
}

set_reminder_schema = {
    "name": "set_reminder",
    "description": """创建一个定时提醒，将在指定时间使用提供的内容通知用户。
    此工具安排在提供的精确时间戳向用户发送通知。
    当用户希望在未来的某个时间点被提醒某事时，应使用此工具。
    提醒系统将存储内容和时间戳，然后在指定时间到达时通过用户首选的通知渠道（移动设备提醒、电子邮件等）触发通知。
    即使应用程序关闭或设备重启，提醒也会持久保存。
    用户可以依赖此功能来处理重要的时间敏感通知，如会议、任务、药物时间表或任何其他有时间限制的活动。
    """,
    "input_schema": {
        "type": "object",
        "properties": {
            "content": {
                "type": "string",
                "description": "将在提醒通知中显示的消息文本。应包含用户希望被提醒的具体信息，例如'服药'、'加入团队视频通话'或'支付水电费'。",
            },
            "timestamp": {
                "type": "string",
                "description": """应触发提醒的确切日期和时间，格式为 ISO 8601 时间戳（YYYY-MM-DDTHH:MM:SS）或 Unix 时间戳。
                系统在内部处理所有时区处理，确保无论用户位于何处，提醒都会在正确的时间触发。
                用户可以简单地指定所需时间，而无需担心时区配置。""",
            },
        },
        "required": ["content", "timestamp"],
    },
}

batch_tool_schema = {
    "name": "batch_tool",
    "description": """同时调用多个其他工具。当你需要执行多个可以并行运行的操作时，使用此工具可以一次性完成所有调用，减少 API 往返次数。
    
    适用场景：
    - 同时设置多个独立的提醒
    - 同时查询多个不相关的数据
    - 执行多个没有依赖关系的操作
    
    注意：只有当操作之间没有依赖关系（即一个操作的输出不需要作为另一个操作的输入）时，才应该使用此工具。""",
    "input_schema": {
        "type": "object",
        "properties": {
            "invocations": {
                "type": "array",
                "description": "要调用的工具调用列表",
                "items": {
                    "type": "object",
                    "properties": {
                        "name": {
                            "type": "string",
                            "description": "要调用的工具名称",
                        },
                        "arguments": {
                            "type": "string",
                            "description": "传递给工具的参数，编码为 JSON 字符串",
                        },
                    },
                    "required": ["name", "arguments"],
                },
            }
        },
        "required": ["invocations"],
    },
}

In [ ]:
import json


def run_tool(tool_name, tool_input):
    print(f"run_tool: {tool_name}, {tool_input}")
    if tool_name == "get_current_datetime":
        return get_current_datetime(**tool_input)
    elif tool_name == "add_duration_to_datetime":
        return add_duration_to_datetime(**tool_input)
    elif tool_name == "set_reminder":
        return set_reminder(**tool_input)
    elif tool_name == "batch_tool":
        return run_batch(**tool_input)


def run_batch(invocations=[]):
    """
    批量工具，同时调用多个其他工具.
    实现后，当 Claude 识别到可以并行执行多个操作时，它将更倾向于使用批量工具。
    Claude 不会发起单独的请求，而是会调用批量工具并传入它想要执行的所有操作列表。

    批量工具本质上是一种变通方法，为 Claude 提供了更高层次的并行工具执行抽象。
    虽然这种方法看起来可能有些不寻常，但它是一种有效的技术，
    可以鼓励 Claude 将相关操作批量处理，减少 API 往返次数并提高整体效率。
    """
    
    batch_output = []
  
    for invocation in invocations:
        name = invocation["name"]
        args = json.loads(invocation["arguments"])
  
        tool_output = run_tool(name, args)
  
        batch_output.append({
            "tool_name": name,
            "output": tool_output
        })
  
    return batch_output


def run_tools(message):
    tool_requests = [block for block in message.content if block.type == "tool_use"]
    tool_result_blocks = []

    for tool_request in tool_requests:
        try:
            tool_output = run_tool(tool_request.name, tool_request.input)
            tool_result_block = {
                "type": "tool_result",
                "tool_use_id": tool_request.id,
                "content": json.dumps(tool_output),
                "is_error": False,
            }
        except Exception as e:
            tool_result_block = {
                "type": "tool_result",
                "tool_use_id": tool_request.id,
                "content": f"Error: {e}",
                "is_error": True,
            }

        tool_result_blocks.append(tool_result_block)

    return tool_result_blocks

In [69]:
def run_conversation(messages):
    while True:
        response = chat(
            messages,
            tools=[
                get_current_datetime_schema,
                add_duration_to_datetime_schema,
                set_reminder_schema,
                batch_tool_schema, # 批量工具,
            ],
        )

        add_assistant_message(messages, response)
        print(text_from_message(response))

        if response.stop_reason != "tool_use":
            break

        tool_results = run_tools(response)
        add_user_message(messages, tool_results)

    return messages

In [70]:
# messages = []
# add_user_message(
#     messages,
#     """1.请为我的医生预约设置一个提醒，当前时间之后的第2000天; 
#        2.请为我吃药预约设置一个提醒，当前时间之后的3小时后
#        3.请为我去超市设置一个提醒，当前时间之后的30分钟后
#        """,
# )
# run_conversation(messages)

# 结构化数据工具

In [71]:
article_summary_schema = {
    "name": "article_summary",
    "description": "创建包含关键见解的文章摘要。当你需要生成文章、研究论文或任何文本内容的结构化摘要时，使用此工具。该工具需要文章标题、作者姓名以及内容中最重要的见解或要点列表。每个见解应该是一个简洁的陈述，捕捉文章的重要观点。",
    "input_schema": {
        "type": "object",
        "properties": {
            "title": {
                "type": "string",
                "description": "正在被摘要的文章标题。",
            },
            "author": {
                "type": "string",
                "description": "撰写文章的作者姓名。",
            },
            "key_insights": {
                "type": "array",
                "items": {"type": "string"},
                "description": "文章中最重要要点或见解的列表。每个见解应该是一个完整、简洁的陈述。",
            },
        },
        "required": ["title", "author", "key_insights"],
    },
}


In [79]:
#结构化数据工具

from IPython.display import Markdown

messages = []
add_user_message(
    messages,
    """
    写一篇关于计算机科学的单段学术文章。 
    包含标题和作者姓名。
    """,
)
response = chat(messages)
Markdown(text_from_message(response))

**标题：** 大语言模型在软件工程自动化中的整合与挑战
**作者：** 张华

随着人工智能技术的飞速发展，大语言模型（LLM）已成为推动软件工程领域范式转移的核心驱动力。通过对海量多语言代码库进行深度表征学习，这些模型不仅能够显著提升自动化代码生成的准确性，还能在复杂的漏洞检测、重构建议及软件维护任务中展现出卓越的逻辑推理能力。然而，当前的集成过程仍面临着严峻的技术挑战，包括模型处理长程依赖时的上下文窗口限制、幻觉现象引发的代码逻辑安全性风险，以及生成的片段与现有异构系统架构之间的兼容性瓶颈。因此，未来的研究方向应致力于开发更具领域针对性的轻量化微调技术，并构建结合神经符号逻辑的自动化评估体系，以确保生成式人工智能在关键任务软件开发中的可靠性、可解释性与安全性，从而最终实现人机协作开发模式的深度进化。

In [80]:
messages = []

add_user_message(messages, text_from_message(response))
response = chat(
    messages,
    tools=[article_summary_schema],
    tool_choice={"type": "tool", "name": "article_summary"},
)
response.content[0].input

{'title': '大语言模型在软件工程自动化中的整合与挑战',
 'author': '张华',
 'key_insights': ['大语言模型（LLM）正推动软件工程领域的范式转移，显著提升了代码生成、漏洞检测和重构建议的自动化水平。',
  '当前集成面临的主要技术瓶颈包括：长程依赖下的上下文窗口限制、模型幻觉引发的安全风险，以及生成的代码与异构系统架构的兼容性问题。',
  '未来的研究应侧重于开发领域针对性的轻量化微调技术，以提高模型在特定软件环境下的表现。',
  '需构建结合神经符号逻辑的自动化评估体系，以确保生成式AI在关键任务软件开发中的可靠性、可解释性和安全性。']}